In [1]:
import geemap
import ee
ee.Initialize(project='pyregence-ee')
import scripts.analysis_functions as af
import scripts.fire_info as fi
import scripts.get_image_collections as gic
import scripts.utils as utils
from generate_model_inputs import get_gridmet, get_mtbs_bs, bi_calc, construct_stack

In [25]:
fires = ee.FeatureCollection("projects/pyregence-ee/assets/conus/mtbs/mtbs_perims_date_processed")
conus_bounds = ee.Image("projects/pyregence-ee/assets/conus/landfire/zones_image").geometry()
fires = fires.filterBounds(conus_bounds)

def fire_yr(feat):
    feature = ee.Feature(feat)
    fire_yr = ee.Number.parse(feature.getString('Ig_Date').slice(0,4))
    return feature.set('fire_yr', fire_yr)

fires_fc = fires.map(fire_yr)

fire_late = fires_fc.filter(ee.Filter.eq('fire_yr', 2017)).sort('pre_start',True).first()
fire_mid = fires_fc.filter(ee.Filter.eq('fire_yr', 2012)).sort('pre_start', False).first()
fire_early = fires_fc.filter(ee.Filter.eq('fire_yr', 1996)).sort('pre_start', False).first()
#print(fires_subset.first().getInfo()['properties'])

bi_late = ee.Image(bi_calc(ee.Feature(fire_late)))
bi_mid = ee.Image(bi_calc(ee.Feature(fire_mid)))
bi_early = ee.Image(bi_calc(ee.Feature(fire_early)))

feature = ee.Feature(fire_late)
print(feature.getString('Ig_Date').getInfo())

region, post_start, post_end, pre_start, pre_end = fi.get_fire_info_from_feature(feature) 
print(fire_start.getInfo())
print(fire_end.getInfo())
print(pre_start.getInfo())
print(pre_end.getInfo())

# landsatCol : None, t2, bestls
# None=T1 real time toa, t2=None+ T1 real time toa, bestls=T1 toa
# cloudBustingMethod : None, bust
# None=No cloud masking/busting, bust=cloud busting
# sensor : landsat, sentinel2
sensor = "landsat"

pre_collection = gic.get_image_collection(sensor,region,pre_start,pre_end,landsatCol='bestls',cloudBustingMethod='bust')
pre_img = gic.get_composite(pre_collection,gic.make_pre_composite,pre_start,pre_end)

post_collection = gic.get_image_collection(sensor,region,fire_start,fire_end,landsatCol='bestls',cloudBustingMethod='bust') # changed to bestls and bust like pre_collection (was None and not defined)
post_img = gic.get_composite(post_collection,gic.make_pre_composite,post_start,post_end) #changed to make_pre_composite (the pre doesn't matter)

pre_nbr = af.nbr(pre_img)
post_nbr = af.nbr(post_img)

dnbr = af.dnbr(pre_nbr, post_nbr).select('dNBR')
rdnbr = af.rdnbr(pre_img,post_img).select('RdNBR')

2017-04-01
2018-03-02
2018-05-01
2016-03-02
2016-05-01


In [26]:
Map = geemap.Map()

# Map.addLayer(pre_img.select('date'), {}, 'pre_img')
# Map.addLayer(post_img.select('date'), {}, 'post_img')
# Map.addLayer(ee.Feature(fire_test), {}, 'fire test perim')
Map.centerObject(region, 8)
Map.addLayer(pre_collection, {"bands":["swir2","nir","red"],"min":0,"max":0.848}, 'pre_collection')
Map.addLayer(post_collection, {"bands":["swir2","nir","red"],"min":0,"max":0.848}, 'post_collection')
Map.addLayer(pre_img,{"bands":["swir2","nir","red"],"min":0,"max":0.848},"pre")
Map.addLayer(post_img,{"bands":["swir2","nir","red"],"min":0,"max":0.848},"post")

Map.addLayer(region, {}, 'region')
Map.addLayer(bi_late, {}, 'bi_late')
# Map.addLayer(bi_mid, {}, 'bi_mid')
# Map.addLayer(bi_early, {}, 'bi_early')


# Map.addLayer(gridmet, {}, 'gridmet')
# Map.addLayer(mtbs, {'min':0, 'max':6, 'palette': ["00ff1f","fbff0e","ffbc00","ff0000"]}, 'mtbs')
#"min":0,"max":1500,"palette":["00ff1f","fbff0e","ffbc00","ff0000"]}
# Map.addLayer(fires_bi_imgs, {}, 'dnbr/rdnbr mosaic')
# Map.addLayer(gridmet_imgs, {}, 'gridmet stack')
# Map.addLayer(mtbs_imgs, {'min':0, 'max':6, 'palette': ["00ff1f","fbff0e","ffbc00","ff0000"]}, 'mtbs')
# Map.addLayer(ee.Image(stack), {}, 'stack'))
Map

# Create annual imageCollection stack that for each fire in the MTBS fire perimeter dataset contains:
## inputs
* dnbr
* rdnbr
* gridmet averages for a few variables
* FVT
## to be predicted
* mtbs burn severity 
